In [ ]:
# =========================
# TESS classification with HistGradientBoosting (no scalers, no RF)
# =========================
!pip install -U scikit-learn numpy pandas joblib

import re
import json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, f1_score, balanced_accuracy_score,
    classification_report, confusion_matrix, make_scorer, precision_recall_curve
)
from sklearn.experimental import enable_hist_gradient_boosting  # noqa
from sklearn.ensemble import HistGradientBoostingClassifier
import joblib

In [ ]:
# =========================
# 2) Google Drive mount
# =========================
from google.colab import drive
drive.mount('/content/drive')

# Edit this path to your file in Drive
csv_path = "/content/drive/MyDrive/tess/tessdataset.csv"


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:


# =========================
# 1) Config
# =========================
label_col = "tfopwg_disp"
random_state = 42
test_size = 0.20
keep_asymmetric = False
min_den = 1e-12

# =========================
# 2) Load & basic cleanup
# =========================
df = pd.read_csv(csv_path)
print("Raw shape:", df.shape)

# Drop unnecessary columns
drop_cols = ["toi", "tid", "rastr", "decstr", "toi_created", "rowupdate"]
df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors="ignore")

# Drop *_lim columns if any
lim_cols = [c for c in df.columns if c.endswith("lim")]
if lim_cols:
    df = df.drop(columns=lim_cols)
    print(f"Dropped {len(lim_cols)} *_lim columns")

# =========================
# 3) Uncertainty → relative error features
# =========================
def add_uncertainty_features(df_in, keep_asymmetric=False, min_den=1e-12):
    df_out = df_in.copy()
    err_cols = [c for c in df_out.columns if re.search(r'(?:_)?err[12]$', c)]
    pairs = {}
    for c in err_cols:
        base = re.sub(r'(?:_)?err[12]$', '', c)
        is_e1 = bool(re.search(r'(?:_)?err1$', c))
        d = pairs.setdefault(base, {"e1": None, "e2": None})
        d["e1" if is_e1 else "e2"] = c

    new_cols, drop_cols = {}, []
    for base, d in pairs.items():
        e1, e2 = d["e1"], d["e2"]
        if (e1 is None) or (e2 is None) or (base not in df_out.columns):
            continue
        val = pd.to_numeric(df_out[base], errors="coerce")
        err1 = pd.to_numeric(df_out[e1], errors="coerce").abs()
        err2 = pd.to_numeric(df_out[e2], errors="coerce").abs()
        denom = np.maximum(np.abs(val), min_den)
        abs_err_mean = (err1 + err2) / 2.0
        new_cols[f"{base}_rel_error"] = abs_err_mean / denom
        if keep_asymmetric:
            new_cols[f"{base}_rel_err_upper"] = err1 / denom
            new_cols[f"{base}_rel_err_lower"] = err2 / denom
        drop_cols += [e1, e2]

    for k, v in new_cols.items():
        df_out[k] = v
    df_out = df_out.drop(columns=drop_cols, errors="ignore")
    return df_out, sorted(new_cols.keys())

df, added_unc_cols = add_uncertainty_features(df, keep_asymmetric=keep_asymmetric, min_den=min_den)
print(f"Added relative-error cols: {len(added_unc_cols)}")

# =========================
# 4) Prepare X, y
# =========================
df = df.dropna(subset=[label_col])
y_raw = df[label_col].astype(str).values
X_raw = df.drop(columns=[label_col]).copy()

# Convert to numeric and drop constant/all-NaN columns
X_raw = X_raw.apply(pd.to_numeric, errors="coerce")
X_raw = X_raw.loc[:, X_raw.nunique(dropna=True) > 1]
X_raw = X_raw.loc[:, ~X_raw.isna().all()]

# Encode labels
le = LabelEncoder()
y = le.fit_transform(y_raw)
print("Classes:", list(le.classes_))
print("Final feature count:", X_raw.shape[1])

# Save feature names for FastAPI inference
with open("feature_names.json", "w", encoding="utf-8") as f:
    json.dump(X_raw.columns.tolist(), f)
print(f"Saved {len(X_raw.columns)} features to feature_names.json")

# =========================
# 5) Train/Test split
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y, test_size=test_size, random_state=random_state, stratify=y
)

# =========================
# 6) Build final pipeline (HistGB only)
# =========================
imputer = SimpleImputer(strategy="median")

histgb = HistGradientBoostingClassifier(
    learning_rate=0.001,
    max_depth=8,
    max_iter=600,
    random_state=random_state
)

pipe = Pipeline(steps=[
    ("imputer", imputer),
    ("clf", histgb),
])

# =========================
# 7) Cross-validation
# =========================
scoring = {
    "f1_macro": make_scorer(f1_score, average="macro"),
    "balanced_accuracy": make_scorer(balanced_accuracy_score),
    "accuracy": make_scorer(accuracy_score),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state)

cv_results = cross_validate(
    pipe, X_train, y_train,
    scoring=scoring, cv=cv, n_jobs=-1, return_train_score=False
)

print("\nCross-validation results (HistGradientBoosting):")
print(f"Mean F1_macro: {np.mean(cv_results['test_f1_macro']):.4f}")
print(f"Mean Balanced Accuracy: {np.mean(cv_results['test_balanced_accuracy']):.4f}")
print(f"Mean Accuracy: {np.mean(cv_results['test_accuracy']):.4f}")

# =========================
# 8) Train + evaluate
# =========================
pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

print("\nHeld-out test performance:")
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=le.classes_, zero_division=0))

# =========================
# 9) Save artifacts
# =========================
joblib.dump(pipe, "histgb_best_pipeline.joblib")
joblib.dump(le, "histgb_label_encoder.joblib")

print("\nSaved artifacts:")
print(" - histgb_best_pipeline.joblib")
print(" - histgb_label_encoder.joblib")
print(" - feature_names.json")


ParserError: Error tokenizing data. C error: Expected 1 fields in line 70, saw 65


In [ ]:
import numpy as np
import pandas as pd

def summarize_split(name, y, le=None):
    labels, counts = np.unique(y, return_counts=True)
    classes = le.inverse_transform(labels) if le is not None else labels
    out = pd.DataFrame({"class": classes, "count": counts})
    out["percent"] = (out["count"] / out["count"].sum() * 100).round(2)
    print(f"\n{name} class distribution:")
    print(out.sort_values("count", ascending=False).reset_index(drop=True))

# Full set
summarize_split("All", y, le)

# If you have splits
summarize_split("Train", y_train, le)
summarize_split("Test", y_test, le)



All class distribution:
  class  count  percent
0    PC   4679    60.74
1    FP   1197    15.54
2    CP    684     8.88
3    KP    583     7.57
4   APC    462     6.00
5    FA     98     1.27

Train class distribution:
  class  count  percent
0    PC   3743    60.74
1    FP    958    15.55
2    CP    547     8.88
3    KP    466     7.56
4   APC    370     6.00
5    FA     78     1.27

Test class distribution:
  class  count  percent
0    PC    936    60.74
1    FP    239    15.51
2    CP    137     8.89
3    KP    117     7.59
4   APC     92     5.97
5    FA     20     1.30
